# 1A pilot v4 — Do missing Vietnamese diacritics hurt LLMs, and why?

> **Tóm tắt (VI):** Đây là phiên bản làm lại từ đầu, rút kinh nghiệm từ mọi lỗi của v1–v3. Notebook chạy **từng bước**; mỗi bước giải thích việc nó làm, in ra thứ cần nhìn, và có **kiểm tra tự động** (✅ / ❌). Nếu một kiểm tra thất bại, notebook dừng lại thay vì cho ra số sai. Cuối notebook là bảng, hình (PNG + PDF) và phần kết luận tự sinh từ số liệu để bạn viết báo.

**Research questions**
1. **RQ1 – Effect:** how much accuracy do open 1.5–8B LLMs lose when Vietnamese text loses its diacritics (partly or fully), and when it is written in teen code?
2. **RQ2 – Who:** does the loss shrink with model size inside a family, and do Vietnamese-adapted models lose less?
3. **RQ3 – Why:** is the loss explained by the **tokenizer** (more, broken tokens) or by **exposure** (the model finds unaccented text surprising)?

## What v4 fixes (lessons from v1–v3)

| Problem found in earlier versions | v4 design |
|---|---|
| Label words tokenized differently per model (Phi: 0% label mass); multi-token labels; Qwen3/bloomz put ~0 probability on the label words | **Multiple-choice letters.** Options are listed as A/B/C…; the answer is **one letter = one token**, checked for every model before running |
| Each model wanted a different answer format (markdown, English, empty) | **Answer prefill chosen per (model, task) on the dev split**, label-blind (highest letter mass), then frozen for the test run |
| VinaLLaMA-2.7b ran without its chat format while the 7b used ChatML | Every model gets **its documented chat format** (own template; ChatML for VinaLLaMA, as on its model card; plain prompt for bloomz, which is not a chat model). Shown and checked in Step 3 |
| Calibration used the test set's own noisy predictions (transductive) | **Calibration from the clean dev split only**, applied unchanged to every condition, as a deployed system would be. Raw and per-condition variants are reported as robustness checks |
| Format choice and calibration were fitted on test items | **Dev (validation split) and test are disjoint**, checked |
| "Perplexity" was computed on a separate prompt; ViMMRC NLL was dominated by the article | NLL is read **from the same forward pass** that scores the answer, **per input field**, in the exact context the model sees |
| Items the noise does not change (ViHSD comments already without diacritics) diluted the effect | Items with no diacritic, duplicates, and texts with conflicting labels are **removed before sampling** |
| Results were reported for models that were not doing the task | **Validity gate decided on clean data before looking at noise:** letter mass ≥ 0.5, free-generation agreement ≥ 0.7, clean accuracy CI above chance |
| No uncertainty, many comparisons | **Paired bootstrap CIs** everywhere; **Holm correction** for the family-size comparisons; SEs clustered by article / premise |

## How to run

**Runtime: L4 GPU** (`Runtime → Change runtime type → L4`). Then `Runtime → Run all`, allow Google Drive.
- Steps 1–3 need only a CPU and finish in a few minutes. Step 4 runs the models: **about 3–4 hours (≈15–20 CU on L4)**. Every finished (model, task) is saved to Drive, so if Colab disconnects, just **Run all again**: finished parts are skipped.
- Steps 5–10 are analysis only. Once Step 4 is complete you can switch to a **CPU runtime** and Run all: the model step is skipped because its results exist.
- Gated models (Llama) need a Hugging Face token in Colab's 🔑 **Secrets** panel, named `HF_TOKEN`. ⚠️ **Never paste a token into a cell**: notebooks get saved and shared.

**Outputs:** `MyDrive/llm-research/1A-pilot-v4/` → `data/`, `preds/`, `figures/` (PNG + PDF), `tables/` (CSV + LaTeX), `checks/`, `key_numbers.json`, `conclusions.md`.

## Step 0 · Setup

Settings you may want to change are all in the second cell. Everything else follows from them.

In [ ]:
import sys, os, re, json, ast, math, random, unicodedata, difflib, warnings, shutil, time, gc
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install -U transformers accelerate
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research"
else:
    ROOT = os.environ.get("PILOT_ROOT", "outputs")
SMOKE = bool(os.environ.get("PILOT_SMOKE"))   # local test only: tiny samples, no validity gate
OUT = f"{ROOT}/1A-pilot-v4" + ("-smoke" if SMOKE else "")
for d in ["data", "preds", "probe", "gen", "figures", "tables", "checks"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
LOCAL_MODELS = json.loads(os.environ.get("PILOT_LOCAL_MODELS", "{}"))  # local test only: {"hub name": "folder"}

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="An input array is constant")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print("results folder:", OUT)

CHECKS = []   # every automatic check, saved to checks/checks.csv
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)[:300]})
    print(("✅ " if ok else "❌ ") + name + (f" — {detail}" if detail else ""))
    pd.DataFrame(CHECKS).drop_duplicates("check", keep="last").to_csv(f"{OUT}/checks/checks.csv", index=False)
    return ok

def require(name, ok, detail=""):
    # a failed requirement stops the notebook: no numbers are better than wrong numbers
    if not check(name, ok, detail):
        raise RuntimeError(f"Stopped: {name}. {detail}")

In [ ]:
SEED = 42
TASKS = ["vsfc", "vsmec", "vihsd", "xnli", "vimmrc"]
CONDITIONS = ["clean", "partial25", "partial50", "partial75", "strip", "teencode", "mixed"]
NOISY = CONDITIONS[1:]
LEVEL = {"clean": 0, "partial25": 25, "partial50": 50, "partial75": 75, "strip": 100}

# items per class: test = the evaluation, dev = format choice and calibration (validation split, disjoint)
TEST_PER_CLASS = {"vsfc": 150, "vsmec": 35, "vihsd": 150, "xnli": 150, "vimmrc": 60}
DEV_PER_CLASS = {"vsfc": 30, "vsmec": 15, "vihsd": 30, "xnli": 30, "vimmrc": 20}
MIN_SYLLABLES = 3

MODEL_INFO = {  # hub name: (family, size in B, Vietnamese-adapted?)   small to large, so cheap ones finish first
    "Qwen/Qwen2.5-1.5B-Instruct": ("Qwen2.5", 1.5, False),
    "Qwen/Qwen3-1.7B": ("Qwen3", 1.7, False),
    "SeaLLMs/SeaLLMs-v3-1.5B-Chat": ("SeaLLMs-v3", 1.5, True),
    "bigscience/bloomz-1b7": ("bloomz", 1.7, False),
    "vilm/vinallama-2.7b-chat": ("VinaLLaMA", 2.7, True),
    "Qwen/Qwen2.5-3B-Instruct": ("Qwen2.5", 3.1, False),
    "bigscience/bloomz-3b": ("bloomz", 3.0, False),
    "Qwen/Qwen3-4B": ("Qwen3", 4.0, False),
    "microsoft/Phi-3.5-mini-instruct": ("Phi-3.5", 3.8, False),
    "Qwen/Qwen2.5-7B-Instruct": ("Qwen2.5", 7.6, False),
    "Qwen/Qwen3-8B": ("Qwen3", 8.2, False),
    "SeaLLMs/SeaLLMs-v3-7B-Chat": ("SeaLLMs-v3", 7.6, True),
    "vilm/vinallama-7b-chat": ("VinaLLaMA", 6.9, True),
    "bigscience/bloomz-7b1": ("bloomz", 7.1, False),
    "mistralai/Mistral-7B-Instruct-v0.3": ("Mistral", 7.2, False),
}
GATED_INFO = {"meta-llama/Llama-3.2-3B-Instruct": ("Llama-3", 3.2, False)}   # needs the HF_TOKEN secret

# Validity gate, fixed before any noisy result is seen (Step 5)
MIN_LETTER_MASS = 0.5     # median probability on the option letters, clean test
MIN_GEN_AGREEMENT = 0.7   # free greedy answer == scored answer
MIN_ABOVE_CHANCE = 0.10   # clean accuracy − chance, and its 95% CI lower bound must be > 0
B = 2000                  # bootstrap resamples
TOKEN_BUDGET = 4096       # padded tokens per forward batch (halved automatically on out-of-memory)
N_GEN = 40                # items per task × {clean, strip} for the free-generation check
if SMOKE:
    TEST_PER_CLASS = {t: 4 for t in TASKS}
    DEV_PER_CLASS = {t: 2 for t in TASKS}
    MIN_LETTER_MASS, MIN_GEN_AGREEMENT, MIN_ABOVE_CHANCE, B, N_GEN = 0.0, 0.0, -1.0, 200, 4

In [ ]:
# Figure style: one colour per model family, the same in every figure
FAMILIES = ["Qwen2.5", "Qwen3", "SeaLLMs-v3", "VinaLLaMA", "bloomz", "Phi-3.5", "Mistral", "Llama-3"]
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FAM_COLOR = dict(zip(FAMILIES, PALETTE))
COND_COLOR = {"partial25": "#cde2fb", "partial50": "#86b6ef", "partial75": "#3987e5", "strip": "#0d366b",
              "teencode": "#eb6834", "mixed": "#6250d6"}
TASK_NAME = {"vsfc": "VSFC (sentiment)", "vsmec": "VSMEC (emotion)", "vihsd": "ViHSD (hate speech)",
             "xnli": "XNLI-vi (inference)", "vimmrc": "ViMMRC (reading)"}
SEQ = mpl.colors.LinearSegmentedColormap.from_list("blue", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#2b2a28",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5, "pdf.fonttype": 42,
})
FIGS = []
def save(fig, name, caption):
    fig.savefig(f"{OUT}/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUT}/figures/{name}.pdf", bbox_inches="tight")
    FIGS.append((name, caption))
    plt.show()

def save_table(df, name, caption, floatfmt="%.2f"):
    df.to_csv(f"{OUT}/tables/{name}.csv")
    try:
        df.to_latex(f"{OUT}/tables/{name}.tex", float_format=floatfmt, caption=caption, label=f"tab:{name}", na_rep="--")
    except Exception as e:
        print("latex export failed:", name, repr(e)[:100])

KEY = {}   # headline numbers → key_numbers.json and the conclusions

## Step 1 · Data: load, clean, sample

For each task: the **validation split** becomes *dev* (used only to choose the answer format and to calibrate) and the **test split** becomes *test* (all reported numbers). Before sampling, every split is cleaned:
- Unicode NFC, whitespace collapsed;
- texts appearing more than once (or with conflicting labels) dropped;
- items with fewer than 3 syllables, or **without any diacritic** (stripping would not change them), dropped;
- dev items whose text also appears in test dropped.

Then a **balanced** sample per class (chance = 1 / number of classes). Option labels are shown to the model as A, B, C, … in a fixed order.

✏️ The Vietnamese option wording and instructions below are worth a native-speaker check.

In [ ]:
PARQUET = "https://huggingface.co/datasets/{}/resolve/refs%2Fconvert%2Fparquet/{}/{}/0000.parquet"
LETTERS = "ABCDEFG"

TASK_SPECS = {
    "vsfc": dict(source=("uitnlp/vietnamese_students_feedback", "default"), label_col="sentiment", classes=[0, 1, 2],
                 labels=["tiêu cực", "trung tính", "tích cực"],
                 fields=[("sentence", "Nhận xét: ")],
                 intro="Đọc nhận xét sau của một sinh viên về môn học và cho biết thái độ của nhận xét.",
                 labels_en=["negative", "neutral", "positive"], fields_en=["Feedback: "],
                 intro_en="Read the following student feedback about a course and state its attitude."),
    "vsmec": dict(source=("ura-hcmut/UIT-VSMEC", "default"), label_col="Emotion",
                  classes=["Enjoyment", "Sadness", "Anger", "Fear", "Disgust", "Surprise", "Other"],
                  labels=["vui vẻ", "buồn bã", "tức giận", "sợ hãi", "ghê tởm", "ngạc nhiên", "khác"],
                  fields=[("Sentence", "Bình luận: ")],
                  intro="Đọc bình luận mạng xã hội sau và cho biết cảm xúc chính mà nó thể hiện.",
                  labels_en=["enjoyment", "sadness", "anger", "fear", "disgust", "surprise", "other"], fields_en=["Comment: "],
                  intro_en="Read the following social media comment and state the main emotion it expresses."),
    "vihsd": dict(source=("ura-hcmut/UIT-ViHSD", "default"), label_col="label_id", classes=[0, 1, 2],
                  labels=["bình thường (không có lời lẽ xấu)", "xúc phạm (lời lẽ thô tục nhưng không nhắm vào ai)",
                          "thù ghét (công kích, lăng mạ một người hay một nhóm người)"],
                  fields=[("free_text", "Bình luận: ")],
                  intro="Đọc bình luận mạng xã hội sau và phân loại nó.",
                  labels_en=["clean (no bad language)", "offensive (vulgar but not aimed at anyone)",
                             "hate (attacks or insults a person or a group)"], fields_en=["Comment: "],
                  intro_en="Read the following social media comment and classify it."),
    "xnli": dict(source=("facebook/xnli", "vi"), label_col="label", classes=[0, 1, 2],
                 labels=["kéo theo (giả thuyết chắc chắn đúng)", "trung lập (không xác định được)",
                         "mâu thuẫn (giả thuyết chắc chắn sai)"],
                 fields=[("premise", "Tiền đề: "), ("hypothesis", "Giả thuyết: ")],
                 intro="Cho một tiền đề và một giả thuyết. Hãy cho biết quan hệ giữa chúng.",
                 labels_en=["entailment (the hypothesis is definitely true)", "neutral (cannot be determined)",
                            "contradiction (the hypothesis is definitely false)"], fields_en=["Premise: ", "Hypothesis: "],
                 intro_en="Given a premise and a hypothesis, state the relation between them."),
    "vimmrc": dict(source=("ura-hcmut/ViMMRC", "default"), label_col="answer", classes=["A", "B", "C", "D"],
                   labels=None,   # the options are part of the item
                   fields=[("article", "Đoạn văn: "), ("question", "Câu hỏi: "), ("options", "Các lựa chọn:\n")],
                   intro="Đọc đoạn văn rồi trả lời câu hỏi trắc nghiệm.",
                   labels_en=None, fields_en=["Passage: ", "Question: ", "Options:\n"],
                   intro_en="Read the passage, then answer the multiple-choice question."),
}
K = {t: len(TASK_SPECS[t]["classes"]) for t in TASKS}
CHANCE = {t: 1 / K[t] for t in TASKS}

def nfc(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()

def strip_diacritics(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s)

def syllable_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text) if any(ch.isalpha() for ch in m.group())]

def accent_share(text):
    w = [x for x in text.split() if any(c.isalpha() for c in x)]
    return sum(strip_diacritics(x) != x for x in w) / max(len(w), 1)

def load_split(task, split):
    spec = TASK_SPECS[task]
    raw = pd.read_parquet(PARQUET.format(*spec["source"], split))
    if task == "vimmrc":
        # options as one field, one option per line; the letters are part of the text the model reads
        raw["options"] = ["\n".join(f"{l}. {nfc(o)}" for l, o in zip("ABCD", ast.literal_eval(x) if isinstance(x, str) else x))
                          for x in raw.options]
    fields = [f for f, _ in spec["fields"]]
    for f in fields:
        raw[f] = raw[f].map(nfc) if f != "options" else raw[f].map(lambda s: "\n".join(nfc(x) for x in s.split("\n")))
    raw = raw[raw[spec["label_col"]].isin(spec["classes"])].copy()
    raw["key"] = raw[fields].agg("\n".join, axis=1)
    n0 = len(raw)
    dup = raw.key.map(raw.key.value_counts()) > 1
    noisy_src = raw[[f for f in fields if f != "options"]].agg(" ".join, axis=1)
    short = noisy_src.map(lambda s: len(syllable_spans(s))) < MIN_SYLLABLES
    no_diac = raw.key.map(strip_diacritics) == raw.key
    keep = raw[~dup & ~short & ~no_diac].copy()
    log = {"task": task, "split": split, "rows": n0, "duplicates": int(dup.sum()), "short": int(short.sum()),
           "no_diacritics": int(no_diac.sum()), "kept": len(keep)}
    return keep, log

def sample(df, task, per_class, seed):
    spec = TASK_SPECS[task]
    n_min = df[spec["label_col"]].value_counts().min()
    require(f"{task}: enough items per class", n_min >= per_class, f"smallest class {n_min} < {per_class}")
    s = df.groupby(spec["label_col"]).sample(n=per_class, random_state=seed).sample(frac=1, random_state=seed)
    s = s.reset_index(drop=True)
    s["label"] = s[spec["label_col"]].map(spec["classes"].index)
    s["item_id"] = range(len(s))
    return s

RAW, DEV, TEST, clean_log = {}, {}, {}, []
for t in TASKS:
    test, lt = load_split(t, "test")
    dev, ld = load_split(t, "validation")
    before = len(dev)
    dev = dev[~dev.key.isin(set(test.key))]
    ld["overlap_with_test"] = before - len(dev)
    clean_log += [lt, ld]
    RAW[t] = test
    TEST[t] = sample(test, t, TEST_PER_CLASS[t], SEED)
    DEV[t] = sample(dev, t, DEV_PER_CLASS[t], SEED + 1)
clean_log = pd.DataFrame(clean_log).set_index(["task", "split"])
save_table(clean_log, "table_data_cleaning", "Rows removed before sampling.", "%d")
display(clean_log)
for t in TASKS:
    require(f"{t}: dev and test share no text", not set(DEV[t].key) & set(TEST[t].key))
    require(f"{t}: test sample is balanced", TEST[t].label.value_counts().nunique() == 1)

In [ ]:
# What the data look like
rows = []
for t in TASKS:
    d = TEST[t]
    fields = [f for f, _ in TASK_SPECS[t]["fields"]]
    rows.append({"task": t, "classes": K[t], "test items": len(d), "dev items": len(DEV[t]), "chance": CHANCE[t],
                 "median syllables": d.key.map(lambda s: len(syllable_spans(s))).median(),
                 "accent share (original)": d.key.map(accent_share).mean()})
data_tab = pd.DataFrame(rows).set_index("task")
save_table(data_tab, "table_datasets", "Tasks: balanced test samples; dev = validation split.")
display(data_tab.round(2))
for t in TASKS:
    r = TEST[t].iloc[0]
    print(f"── {t} (label {r.label})\n   " + " | ".join(f"{f}: {r[f][:90]!r}" for f, _ in TASK_SPECS[t]["fields"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
L = {t: TEST[t].key.map(lambda s: len(syllable_spans(s))) for t in TASKS}
axes[0].boxplot([L[t] for t in TASKS], vert=False, widths=0.5, showfliers=False,
                medianprops=dict(color="#2a78d6", lw=2), boxprops=dict(color="#52514e"),
                whiskerprops=dict(color="#52514e"), capprops=dict(color="#52514e"))
axes[0].set(xscale="log", yticks=range(1, len(TASKS) + 1), yticklabels=[TASK_NAME[t] for t in TASKS],
            xlabel="syllables per item (log scale)", title="a. Item length")
axes[0].invert_yaxis()
for i, t in enumerate(TASKS):
    axes[1].hist(TEST[t].key.map(accent_share), bins=np.linspace(0, 1, 21), histtype="step", lw=2, color=PALETTE[i],
                 label=t, density=True)
axes[1].set(xlabel="share of syllables carrying a diacritic", ylabel="density", title="b. Diacritics in the original text")
axes[1].legend(fontsize=7)
plt.tight_layout()
save(fig, "fig01_data", "Test items: length and how much of the original text carries diacritics (ViHSD is social media and already partly unaccented).")

## Step 2 · Noise: build and verify

Seven conditions, applied to the **input fields only** (instructions and option labels stay clean, except ViMMRC whose options are part of the item):

| condition | what it does |
|---|---|
| `clean` | original text |
| `partial25/50/75` | remove the diacritics of 25/50/75% of the words that have one (random, seeded per item) |
| `strip` | remove every diacritic (đ → d) |
| `teencode` | common Vietnamese chat abbreviations (không → ko, được → dc, …), diacritics kept |
| `mixed` | teen code, then 50% of diacritics removed |

Checks below: `strip` leaves no diacritic, `partial*` changes nothing but diacritics and hits the intended rate, every condition is deterministic.

In [ ]:
def partial_strip(text, rate, rng):
    words = text.split(" ")
    cand = [i for i, w in enumerate(words) if strip_diacritics(w) != w]
    chosen = set(rng.sample(cand, round(rate * len(cand))))
    return " ".join(strip_diacritics(w) if i in chosen else w for i, w in enumerate(words))

TEENCODE = {
    "giáo viên": "gv", "giảng viên": "gv", "sinh viên": "sv", "bài tập": "bt", "học sinh": "hs",
    "không": "ko", "được": "dc", "gì": "j", "với": "vs", "mình": "mk", "rồi": "r",
    "vậy": "v", "bạn": "bn", "quá": "wa", "nhiều": "nhìu", "người": "ng", "trước": "trc",
    "cũng": "cx", "thôi": "thui", "biết": "bít", "yêu": "iu", "giờ": "h",
    "nhưng": "nhg", "lắm": "lém", "thích": "thik", "những": "nhữg", "hiểu": "hỉu",
}
_teen_re = re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, TEENCODE), key=len, reverse=True)) + r")(?!\w)",
                      re.IGNORECASE)

def teencode(text):
    return _teen_re.sub(lambda m: TEENCODE[m.group(0).lower()], text)

def make_noisy(text, cond, item_key, field):
    rng = random.Random(f"{SEED}|{item_key}|{cond}|{field}")
    if cond == "clean":
        return text
    if cond == "strip":
        return strip_diacritics(text)
    if cond.startswith("partial"):
        return partial_strip(text, int(cond[len("partial"):]) / 100, rng)
    if cond == "teencode":
        return teencode(text)
    if cond == "mixed":
        return partial_strip(teencode(text), 0.5, rng)
    raise ValueError(cond)

def expand(df, task, conds):
    fields = [f for f, _ in TASK_SPECS[task]["fields"]]
    rows = []
    for r in df.itertuples(index=False):
        r = r._asdict()
        for c in conds:
            row = {"task": task, "item_id": r["item_id"], "label": r["label"], "cond": c}
            for f in fields:
                noisy = r[f] if f == "options" and c == "clean" else make_noisy(r[f], c, r["key"], f)
                row[f], row[f + "_clean"] = noisy, r[f]
            row["text"] = "\n".join(row[f] for f in fields)
            row["clean"] = "\n".join(row[f + "_clean"] for f in fields)
            rows.append(row)
    return pd.DataFrame(rows)

DATA = {t: expand(TEST[t], t, CONDITIONS) for t in TASKS}
DEVDATA = {t: expand(DEV[t], t, ["clean"]) for t in TASKS}
pd.concat(DATA.values())[["task", "item_id", "label", "cond", "text", "clean"]].to_csv(f"{OUT}/data/test_items.csv", index=False)
pd.concat(DEVDATA.values())[["task", "item_id", "label", "text"]].to_csv(f"{OUT}/data/dev_items.csv", index=False)

again = expand(TEST["vsfc"], "vsfc", CONDITIONS)
require("noise is deterministic", again.text.equals(DATA["vsfc"].text))
assert strip_diacritics("Tôi muốn đặt vé máy bay") == "Toi muon dat ve may bay"

rows = []
for t in TASKS:
    d = DATA[t]
    for c in NOISY:
        g = d[d.cond == c]
        row = {"task": t, "cond": c, "items changed": (g.text != g.clean).mean(),
               "syllables changed": np.mean([1 - difflib.SequenceMatcher(None, a.split(), b.split(), autojunk=False).ratio()
                                             for a, b in zip(g.clean, g.text)])}
        if c in LEVEL:
            row["only diacritics changed"] = (g.text.map(strip_diacritics) == g.clean.map(strip_diacritics)).mean()
        rows.append(row)
noise_tab = pd.DataFrame(rows)
noise_tab.to_csv(f"{OUT}/checks/noise.csv", index=False)
st = noise_tab[noise_tab.cond == "strip"]
require("strip changes every test item", (st["items changed"] == 1).all(), st.set_index("task")["items changed"].round(3).to_dict())
require("strip/partial change only diacritics", (noise_tab.dropna()["only diacritics changed"] == 1).all())
require("strip leaves no diacritic", all((DATA[t][DATA[t].cond == "strip"].text.map(strip_diacritics) ==
                                          DATA[t][DATA[t].cond == "strip"].text).all() for t in TASKS))
hit = tot = 0
for t in TASKS:
    g = DATA[t][DATA[t].cond == "partial50"]
    for r in g.itertuples():
        for f, _ in TASK_SPECS[t]["fields"]:
            a, b = getattr(r, f + "_clean").split(" "), getattr(r, f).split(" ")
            cand = [i for i, w in enumerate(a) if strip_diacritics(w) != w]
            hit += sum(a[i] != b[i] for i in cand)
            tot += len(cand)
check("partial50 strips ~50% of accented words", abs(hit / tot - 0.5) < 0.02, f"realized {hit / tot:.3f}")
display(noise_tab.pivot(index="task", columns="cond", values="items changed")[NOISY].round(2))
ex = DATA["vsfc"][DATA["vsfc"].item_id == 0].set_index("cond").text.reindex(CONDITIONS).to_frame("example (VSFC)")
save_table(ex, "table_noise_example", "One VSFC test item under every condition (original Vietnamese text).")
display(ex)

In [ ]:
# Lexical ambiguity: how many accented words collapse onto the same unaccented form (information destroyed by stripping)
lex = pd.Series([w.lower() for t in TASKS for f, _ in TASK_SPECS[t]["fields"] for s in RAW[t][f]
                 for w in re.findall(r"[^\W\d_]+", s)])
FORMS = lex.groupby(lex.map(strip_diacritics)).nunique()
print(f"lexicon from the test splits: {lex.nunique()} word types → {len(FORMS)} unaccented forms; "
      f"{(FORMS > 1).mean():.0%} of unaccented forms are ambiguous")

def ambiguity_bits(clean, noisy):
    a, b = clean.split(" "), noisy.split(" ")
    if len(a) != len(b):
        return np.nan
    ws = [re.sub(r"[^\w]", "", y.lower()) for x, y in zip(a, b) if x != y]
    n = np.array([FORMS.get(w, 1) for w in ws if w])
    return np.log2(n).mean() if len(n) else np.nan

FEAT = []
for t in TASKS:
    unit_field = {"vimmrc": "article_clean", "xnli": "premise_clean"}.get(t)
    for r in DATA[t].itertuples():
        FEAT.append({"task": t, "item_id": r.item_id, "cond": r.cond,
                     "changed": 1 - difflib.SequenceMatcher(None, r.clean.split(), r.text.split(), autojunk=False).ratio(),
                     "amb_bits": ambiguity_bits(r.clean, r.text) if r.cond in LEVEL and r.cond != "clean" else np.nan,
                     "unit": f"{t}|" + (getattr(r, unit_field) if unit_field else str(r.item_id))})
FEAT = pd.DataFrame(FEAT)
FEAT["unit"] = pd.factorize(FEAT.unit)[0]   # clustering unit: article (ViMMRC), premise (XNLI), item (others)

fig, ax = plt.subplots(figsize=(6.2, 3))
x = np.arange(len(TASKS))
for j, c in enumerate(NOISY):
    v = FEAT[FEAT.cond == c].groupby("task").changed.mean().reindex(TASKS)
    ax.bar(x + (j - 2.5) * 0.13, v, 0.12, color=COND_COLOR[c], label=c)
ax.set(xticks=x, xticklabels=TASKS, ylabel="share of words changed", title="How much each condition changes the text")
ax.legend(fontsize=7, ncol=3)
save(fig, "fig02_noise_dose", "Share of words each condition changes, per task.")

## Step 3 · Models and prompts: look before running (CPU)

For every model, using its tokenizer only:
1. **Prompt format**: the model's own chat template; VinaLLaMA-2.7b ships without one, so it gets the ChatML template from VinaLLaMA-7b (both model cards document ChatML); bloomz is not a chat model and gets a plain prompt. The exact prompt for one item is printed.
2. **Answer letters**: each option letter must be **one token** right after the prompt, and the letters must be different tokens. Checked for every prefill candidate.
3. **Length**: the longest prompt must fit the model's context.
4. **Tokenizer statistics** of the input fields *inside the prompt* (what the model actually reads): tokens per syllable, share of syllables split into several tokens.

A model that fails a check is listed and left out; nothing runs on a broken setup.

In [ ]:
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
MODEL_INFO.update(GATED_INFO if (HF_TOKEN or any(m in LOCAL_MODELS for m in GATED_INFO)) else {})
print("gated models:", "included" if any(m in MODEL_INFO for m in GATED_INFO) else "skipped (no HF_TOKEN secret)")
SHORT = {m: m.split("/")[1].replace("-Instruct", "").replace("-Chat", "").replace("-chat", "").replace("-mini-instruct", "-mini")
         .replace("-v0.3", "") for m in MODEL_INFO}

from transformers import AutoTokenizer
CHAT_TEMPLATE_FROM = {"vilm/vinallama-2.7b-chat": "vilm/vinallama-7b-chat"}   # documented ChatML, missing in the 2.7b tokenizer
PREFILLS = {"chat": ["Đáp án:", "", "**"], "plain": ["Đáp án:", ""]}

def load_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(name, name), token=HF_TOKEN)
    source = "own chat template" if tok.chat_template else "plain prompt (not a chat model)"
    if name in CHAT_TEMPLATE_FROM:
        donor = CHAT_TEMPLATE_FROM[name]
        tok.chat_template = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(donor, donor), token=HF_TOKEN).chat_template
        source = f"ChatML from {donor.split('/')[1]}"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok, source

def build_message(task, row):
    # returns the user message and the character span of every input field inside it
    spec = TASK_SPECS[task]
    parts = [(spec["intro"] + "\n\n", None)]
    for f, prefix in spec["fields"]:
        parts += [(prefix, None), (row[f], f), ("\n", None)]
    if spec["labels"]:
        parts.append(("\nCác lựa chọn:\n" + "\n".join(f"{L}. {o}" for L, o in zip(LETTERS, spec["labels"])) + "\n", None))
    letters = LETTERS[:K[task]]
    parts.append(("\nChỉ trả lời bằng một chữ cái: " + ", ".join(letters[:-1]) + " hoặc " + letters[-1] + ".", None))
    msg, spans = "", {}
    for text, f in parts:
        if f:
            spans[f] = (len(msg), len(msg) + len(text))
        msg += text
    return msg, spans

_WRAP = {}
def chat_wrapper(tok):
    # the chat template rendered once around a placeholder; the template copies the message verbatim (checked)
    if id(tok) not in _WRAP:
        ph = "<<<MESSAGE>>>"
        w = tok.apply_chat_template([{"role": "user", "content": ph}], tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
        assert w.count(ph) == 1, "chat template does not copy the message verbatim"
        test_msg = "Thử: tiếng Việt có dấu.\nDòng hai."
        full = tok.apply_chat_template([{"role": "user", "content": test_msg}], tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
        assert full == w.replace(ph, test_msg), "chat template changes the message"
        _WRAP[id(tok)] = w.split(ph)
    return _WRAP[id(tok)]

def build_prompt(tok, task, row, prefill):
    msg, spans = build_message(task, row)
    if tok.chat_template:
        head, tail = chat_wrapper(tok)
        off = len(head)
        prompt, add_special = head + msg + tail + prefill, False
    else:
        off, prompt, add_special = 0, msg + "\n" + prefill, True
    return prompt, add_special, {f: (a + off, b + off) for f, (a, b) in spans.items()}

def encode(tok, task, row, prefill):
    # token ids, the token positions of each field, and per-field tokenizer statistics
    prompt, add_special, spans = build_prompt(tok, task, row, prefill)
    enc = tok(prompt, add_special_tokens=add_special, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    pos, stats = {}, {}
    for f, (a, b) in spans.items():
        p = [i for i, (x, y) in enumerate(offs) if y > x and x < b and y > a and i > 0]
        sy = [(a + s, a + e) for s, e in syllable_spans(prompt[a:b])]
        # a syllable is "broken" if more than one token covers it; offsets are sorted, so walk them once
        tok_spans = [offs[i] for i in p]
        j, broken = 0, 0
        for s_, e_ in sy:
            while j < len(tok_spans) and tok_spans[j][1] <= s_:
                j += 1
            k, n = j, 0
            while k < len(tok_spans) and tok_spans[k][0] < e_:
                n += 1
                k += 1
            broken += n > 1
        pos[f] = p
        stats[f] = (len(p), len(sy), broken)
    return prompt, add_special, ids, pos, stats

_LETTERS = {}
def letter_ids(tok, prompt, add_special, ids, task):
    # for each option letter: token ids of " X" / "X" that are ONE token right after the prompt.
    # Every prompt of a task ends with the same text, so the answer is cached on that ending (and re-checked on its tokens).
    key = (id(tok), task, prompt[-80:], tuple(ids[-6:]))
    if key in _LETTERS:
        return _LETTERS[key]
    out = []
    for L in LETTERS[:K[task]]:
        v = []
        for s in (" " + L, L):
            full = tok(prompt + s, add_special_tokens=add_special)["input_ids"]
            if full[:len(ids)] == ids and len(full) == len(ids) + 1:
                v.append(full[-1])
        out.append(sorted(set(v)))
    _LETTERS[key] = out
    return out

def context_len(name, tok):
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(LOCAL_MODELS.get(name, name), token=HF_TOKEN)
    n = getattr(cfg, "max_position_embeddings", None) or getattr(cfg, "seq_length", None) or getattr(cfg, "n_positions", None)
    return n or 2048   # bloom uses ALiBi (no hard limit); 2048 is its training length

In [ ]:
TOK, setup_rows, TOKSTATS = {}, [], []
for name in MODEL_INFO:
    try:
        tok, source = load_tokenizer(name)
    except Exception as e:
        check(f"{SHORT[name]}: tokenizer loads", False, repr(e)[:150])
        continue
    kind = "chat" if tok.chat_template else "plain"
    ok, notes, longest = True, [], 0
    for t in TASKS:
        rows = DATA[t].to_dict("records")
        for pre in PREFILLS[kind]:
            r = rows[0]
            prompt, add, ids, pos, _ = encode(tok, t, r, pre)
            lid = letter_ids(tok, prompt, add, ids, t)
            firsts = [v[0] for v in lid if v]
            if not all(lid) or len(set(firsts)) != len(firsts):
                notes.append(f"{t}/prefill {pre!r}: letters not single distinct tokens")
                if pre == "Đáp án:":
                    ok = False
        for r in rows:   # tokenizer statistics of the input fields, inside the real prompt
            _, _, ids, _, stats = encode(tok, t, r, "Đáp án:")
            longest = max(longest, len(ids))
            TOKSTATS.append({"model": name, "task": t, "item_id": r["item_id"], "cond": r["cond"],
                             **{f"{k}_{f}": v for f, s in stats.items() for k, v in zip(["ntok", "nsyl", "nbroken"], s)}})
    ctx = context_len(name, tok)
    ok &= longest + 16 <= ctx
    TOK[name] = tok
    setup_rows.append({"model": SHORT[name], "family": MODEL_INFO[name][0], "size_B": MODEL_INFO[name][1],
                       "vi_adapted": MODEL_INFO[name][2], "prompt format": source, "longest prompt": longest,
                       "context": ctx, "ok": ok, "notes": "; ".join(notes)})
    check(f"{SHORT[name]}: letters are single tokens and the longest prompt fits", ok, "; ".join(notes) or f"{longest}/{ctx} tokens")
setup = pd.DataFrame(setup_rows).set_index("model")
save_table(setup.drop(columns="notes"), "table_models", "Models, prompt format, and prompt length.")
display(setup)
MODELS = [m for m in MODEL_INFO if m in TOK and setup.loc[SHORT[m], "ok"]]
print(len(MODELS), "models pass the setup checks")
TOKSTATS = pd.DataFrame(TOKSTATS)
TOKSTATS.to_csv(f"{OUT}/data/tokenizer_stats.csv", index=False)

In [ ]:
# The exact input of one model per prompt format, so you can see what the model sees
shown = set()
for name in MODELS:
    tok = TOK[name]
    kind = setup.loc[SHORT[name], "prompt format"]
    if kind in shown:
        continue
    shown.add(kind)
    r = DATA["vsfc"][DATA["vsfc"].cond == "strip"].iloc[0].to_dict()
    prompt, add, ids, pos, _ = encode(tok, "vsfc", r, "Đáp án:")
    print(f"══ {SHORT[name]} — {kind} — {len(ids)} tokens; field tokens: {tok.convert_ids_to_tokens([ids[i] for i in pos['sentence']])[:12]}")
    print(prompt)
    print("letter token ids:", letter_ids(tok, prompt, add, ids, "vsfc"), "\n")

In [ ]:
# Tokenizer behaviour per model: tokens per syllable and broken syllables, clean vs strip (all tasks pooled)
def field_sum(df, kind):
    return df[[c for c in df.columns if c.startswith(kind + "_")]].sum(axis=1)

ts = TOKSTATS.assign(ntok=field_sum(TOKSTATS, "ntok"), nsyl=field_sum(TOKSTATS, "nsyl"), nbroken=field_sum(TOKSTATS, "nbroken"))
g = ts.groupby(["model", "cond"])[["ntok", "nsyl", "nbroken"]].sum()
tokc = pd.DataFrame({"fertility": g.ntok / g.nsyl, "broken": g.nbroken / g.nsyl}).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(10, 0.32 * len(MODELS) + 1.2), sharey=True)
y = np.arange(len(MODELS))
for ax, k, lab in [(axes[0], "fertility", "tokens per syllable"), (axes[1], "broken", "share of syllables split into >1 token")]:
    D = tokc.pivot(index="model", columns="cond", values=k).reindex(MODELS)
    for i, m in enumerate(MODELS):
        ax.plot([D.loc[m, "clean"], D.loc[m, "strip"]], [i, i], color="#cfcec8", lw=2, zorder=1)
    ax.scatter(D["clean"], y, color="#2a78d6", s=28, zorder=2, label="clean")
    ax.scatter(D["strip"], y, color="#eb6834", s=28, zorder=2, label="strip")
    ax.set(xlabel=lab, yticks=y, yticklabels=[SHORT[m] for m in MODELS])
axes[0].invert_yaxis()
axes[0].legend(fontsize=8)
axes[0].set_title("a. Fertility")
axes[1].set_title("b. Broken syllables")
plt.tight_layout()
save(fig, "fig03_tokenizers", "How each tokenizer splits Vietnamese with and without diacritics (input fields, inside the real prompt).")
# models that tokenize every input identically share a tokenizer: used by the same-tokenizer test in Step 8
sig = ts.sort_values(["task", "item_id", "cond"]).groupby("model").ntok.apply(lambda s: hash(tuple(s)))
TOK_CLASS = {m: f"tok{i}" for i, (_, ms) in enumerate(sig.groupby(sig)) for m in ms.index}
print("tokenizer groups:", [[SHORT[m] for m in ms.index] for _, ms in sig.groupby(sig) if len(ms) > 1])

In [ ]:
# The prompts in Vietnamese (as the models see them) with an English translation, for the paper's appendix
def template(t, lang):
    spec = TASK_SPECS[t]
    sfx = "_en" if lang == "en" else ""
    lines = [spec["intro" + sfx], ""]
    lines += [f"{pre.rstrip()} {{{f}}}" for (f, _), pre in zip(spec["fields"], [p for _, p in spec["fields"]] if not sfx else spec["fields_en"])]
    labels = spec["labels" + sfx]
    if labels:
        lines += ["", "Các lựa chọn:" if not sfx else "Options:"] + [f"{L}. {o}" for L, o in zip(LETTERS, labels)]
    letters = LETTERS[:K[t]]
    lines += ["", ("Chỉ trả lời bằng một chữ cái: " if not sfx else "Answer with one letter only: ")
              + ", ".join(letters[:-1]) + (" hoặc " if not sfx else " or ") + letters[-1] + "."]
    return "\n".join(lines)
prompts_tab = pd.DataFrame({"task": [TASK_NAME[t] for t in TASKS], "prompt (Vietnamese, as used)": [template(t, "vi") for t in TASKS],
                            "English translation": [template(t, "en") for t in TASKS]}).set_index("task")
save_table(prompts_tab, "table_prompts", "Prompt templates (user message; Vietnamese as used, with English translation). "
           "The answer prefill \\textit{Đáp án:} means \\textit{Answer:}.")
for t in TASKS:
    print(f"── {TASK_NAME[t]} ──\n{template(t, 'en')}\n")

## Step 4 · Run the models (GPU)

Per model, per task:
1. **Format probe (dev, clean):** score the dev items with each answer prefill (`"Đáp án:"`, none, and for chat models `"**"`); keep the prefill with the highest median **letter mass** (probability on the option letters). The gold labels are never used here.
2. **Calibration prior (dev, clean):** the mean predicted distribution over letters on the dev items. Dividing by it removes a model's preference for some letters (dev is balanced, so a fair model's prior is uniform).
3. **Test run:** all conditions. **One forward pass per prompt** gives both the answer (next-token distribution over the letters) and the NLL of every input token, in context.
4. **Free-generation check:** greedy answers for 40 clean + 40 stripped test items per task, parsed for a letter, compared with the scored answer.

Saved per (model, task): `preds/`, `probe/`, `gen/`. Already-finished parts are skipped.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from huggingface_hub import constants as hf_constants
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (torch.bfloat16 if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8
         else torch.float16 if DEVICE == "cuda" else torch.float32)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| dtype:", DTYPE)

def fname(name):
    return name.replace("/", "__")

def pad_batch(seqs, pad_id):
    Lm = max(map(len, seqs))
    ids = torch.full((len(seqs), Lm), pad_id)
    mask = torch.zeros((len(seqs), Lm), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

@torch.no_grad()
def forward_batch(model, tok, items):
    # items: dicts with ids, letters (list of token-id lists), pos (field → token positions)
    ids, mask = pad_batch([it["ids"] for it in items], tok.pad_token_id)
    logits = model(input_ids=ids, attention_mask=mask).logits
    last = logits[torch.arange(len(items), device=DEVICE), mask.sum(1) - 1].float().log_softmax(-1)
    out = []
    # gather log p(token_t | tokens < t) for every field token, in chunks to keep memory bounded
    pairs = [(b, p) for b, it in enumerate(items) for f in it["pos"] for p in it["pos"][f]]
    tok_lp = {}
    for i in range(0, len(pairs), 512):
        chunk = pairs[i:i + 512]
        bb = torch.tensor([b for b, _ in chunk], device=DEVICE)
        pp = torch.tensor([p for _, p in chunk], device=DEVICE)
        sel = logits[bb, pp - 1].float()
        lp = sel.gather(1, ids[bb, pp][:, None]).squeeze(1) - sel.logsumexp(-1)
        tok_lp.update(zip(chunk, lp.tolist()))
    del logits
    for b, it in enumerate(items):
        letter_lp = [torch.logsumexp(last[b, v], 0).item() for v in it["letters"]]
        nll = {f: -sum(tok_lp[(b, p)] for p in it["pos"][f]) for f in it["pos"]}
        out.append((letter_lp, nll))
    return out

def batches(items, budget):
    order = sorted(range(len(items)), key=lambda i: -len(items[i]["ids"]))
    cur, longest = [], 0
    for i in order:
        n = len(items[i]["ids"])
        if cur and max(longest, n) * (len(cur) + 1) > budget:
            yield cur
            cur, longest = [], 0
        cur.append(i)
        longest = max(longest, n)
    if cur:
        yield cur

def score(model, tok, task, rows, prefill, budget):
    items = []
    for r in rows:
        prompt, add, ids, pos, stats = encode(tok, task, r, prefill)
        items.append({"ids": ids, "pos": pos, "letters": letter_ids(tok, prompt, add, ids, task)})
    res = [None] * len(items)
    for idx in batches(items, budget):
        for i, o in zip(idx, forward_batch(model, tok, [items[i] for i in idx])):
            res[i] = o
    lp = np.array([r[0] for r in res])
    nll = pd.DataFrame([r[1] for r in res]).add_prefix("nll_")
    return lp, nll

def softmax_rows(lp):
    p = np.exp(lp - lp.max(1, keepdims=True))
    return p / p.sum(1, keepdims=True)

@torch.no_grad()
def generate(model, tok, task, rows, prefill, max_new_tokens=12):
    tok.padding_side = "left"
    out = []
    for i in range(0, len(rows), 8):
        enc = [encode(tok, task, r, prefill) for r in rows[i:i + 8]]
        Lm = max(len(e[2]) for e in enc)
        ids = torch.tensor([[tok.pad_token_id] * (Lm - len(e[2])) + e[2] for e in enc], device=DEVICE)
        mask = (torch.arange(Lm)[None, :] >= torch.tensor([Lm - len(e[2]) for e in enc])[:, None]).long().to(DEVICE)
        g = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        out += [tok.decode(x[Lm:], skip_special_tokens=True) for x in g]
    tok.padding_side = "right"
    return out

def parse_letter(text, task):
    m = re.search(r"\b([A-G])\b", text.replace("*", " ")[:40])
    return LETTERS.index(m.group(1)) if m and LETTERS.index(m.group(1)) < K[task] else -1

In [ ]:
def run_task(model, tok, name, task, budget):
    kind = "chat" if tok.chat_template else "plain"
    # 1. format probe on dev (label-blind)
    dev_rows = DEVDATA[task].to_dict("records")
    probe = []
    for pre in PREFILLS[kind]:
        lid = letter_ids(tok, *encode(tok, task, dev_rows[0], pre)[:3], task)
        if not all(lid):
            continue
        lp, _ = score(model, tok, task, dev_rows, pre, budget)
        mass = np.exp(lp).sum(1)
        probe.append({"model": name, "task": task, "prefill": pre, "median_mass": float(np.median(mass)),
                      "dev_acc_raw": float((lp.argmax(1) == DEVDATA[task].label.values).mean()), "lp": lp})
    best = max(probe, key=lambda p: p["median_mass"])
    pre = best["prefill"]
    # 2. calibration prior from the clean dev items, with the chosen prefill
    prior = softmax_rows(best["lp"]).mean(0)
    # 3. test run
    rows = DATA[task].to_dict("records")
    lp, nll = score(model, tok, task, rows, pre, budget)
    p = softmax_rows(lp)
    out = DATA[task][["task", "item_id", "label", "cond"]].reset_index(drop=True)
    out["model"] = name
    for k in range(K[task]):
        out[f"lp{k}"] = lp[:, k]
    out["letter_mass"] = np.exp(lp).sum(1)
    out["pred_raw"] = p.argmax(1)
    out["pred"] = (np.log(p) - np.log(prior)).argmax(1)
    out = pd.concat([out, nll], axis=1)
    out["prefill"] = pre
    # 4. free-generation check
    sub = DATA[task][DATA[task].item_id.isin(DATA[task].item_id.unique()[:N_GEN]) & DATA[task].cond.isin(["clean", "strip"])]
    gen_rows = sub.to_dict("records")
    answers = generate(model, tok, task, gen_rows, pre)
    gen = sub[["task", "item_id", "label", "cond"]].assign(model=name, answer=answers,
                                                           parsed=[parse_letter(a, task) for a in answers])
    gen = gen.merge(out[["item_id", "cond", "pred_raw"]], on=["item_id", "cond"])
    probe_df = pd.DataFrame([{k: v for k, v in pr.items() if k != "lp"} for pr in probe]).assign(
        chosen=lambda d: d.prefill == pre, prior=str(np.round(prior, 3).tolist()))
    return out, probe_df, gen

def done(name, task):
    return all(os.path.exists(f"{OUT}/{d}/{task}__{fname(name)}.csv") for d in ["preds", "probe", "gen"])

todo = [(m, t) for m in MODELS for t in TASKS if not done(m, t)]
print(f"{len(todo)} (model, task) runs to do")
if todo and DEVICE != "cuda" and not SMOKE:
    raise RuntimeError("Model runs are missing and there is no GPU: Runtime → Change runtime type → L4 GPU")
failed = {}
for name in MODELS:
    tasks = [t for t in TASKS if not done(name, t)]
    if not tasks:
        continue
    t0 = time.time()
    try:
        tok = TOK[name]
        model = AutoModelForCausalLM.from_pretrained(LOCAL_MODELS.get(name, name), dtype=DTYPE, token=HF_TOKEN).to(DEVICE).eval()
    except Exception as e:
        failed[name] = repr(e)[:200]
        print("✗ could not load", name, failed[name])
        continue
    print(f"● {SHORT[name]} loaded in {time.time() - t0:.0f}s")
    for t in tasks:
        t1, budget = time.time(), TOKEN_BUDGET
        while True:
            try:
                out, probe_df, gen = run_task(model, tok, name, t, budget)
                break
            except torch.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                budget //= 2
                print(f"   out of memory, token budget → {budget}")
                if budget < 512:
                    out = None
                    break
        if out is None:
            failed[f"{name}/{t}"] = "out of memory"
            continue
        out.to_csv(f"{OUT}/preds/{t}__{fname(name)}.csv", index=False)
        probe_df.to_csv(f"{OUT}/probe/{t}__{fname(name)}.csv", index=False)
        gen.to_csv(f"{OUT}/gen/{t}__{fname(name)}.csv", index=False)
        acc = (out.pred == out.label).groupby(out.cond).mean()
        agree = (gen.parsed == gen.pred_raw).mean()
        print(f"   ✓ {t:7} {time.time() - t1:4.0f}s  prefill={out.prefill[0]!r:10} mass={out[out.cond == 'clean'].letter_mass.median():.2f}"
              f"  clean={acc['clean']:.2f} strip={acc['strip']:.2f} (chance {CHANCE[t]:.2f})  gen-agree={agree:.2f}"
              f"  e.g. {gen.answer.iloc[0][:30]!r}")
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if IN_COLAB:   # 7B weights are ~15 GB; keep Colab's disk free
        shutil.rmtree(os.path.join(hf_constants.HF_HUB_CACHE, "models--" + name.replace("/", "--")), ignore_errors=True)
check("every (model, task) run finished", not failed and all(done(m, t) for m in MODELS for t in TASKS), failed or "")

## Step 5 · Audit the outputs and apply the validity gate

From here on, everything is CPU-only and reads the saved files.

**Integrity:** every file complete, no duplicate rows, finite log-probabilities, stored predictions reproducible from the stored log-probabilities, same items and labels for every model.

**Validity gate** (on *clean* test data only, decided before looking at noise). A (model, task) enters the analysis if:
- median **letter mass ≥ 0.5** (the model answers with a letter at all);
- **free-generation agreement ≥ 0.7** (what it says matches what we score);
- clean accuracy − chance ≥ 0.10 **and** the 95% bootstrap CI of clean accuracy lies above chance.

Excluded pairs are listed with the reason. They are not "bad results": the model does not do the task in this format, so a noise effect on it would be meaningless.

In [ ]:
frames, gens, probes = [], [], []
for m in MODELS:
    for t in TASKS:
        if not done(m, t):
            continue
        frames.append(pd.read_csv(f"{OUT}/preds/{t}__{fname(m)}.csv", keep_default_na=False))
        gens.append(pd.read_csv(f"{OUT}/gen/{t}__{fname(m)}.csv", keep_default_na=False))
        probes.append(pd.read_csv(f"{OUT}/probe/{t}__{fname(m)}.csv", keep_default_na=False))
P = pd.concat(frames, ignore_index=True)
GEN = pd.concat(gens, ignore_index=True)
PROBE = pd.concat(probes, ignore_index=True)
MODELS = [m for m in MODELS if m in set(P.model)]
P["family"] = P.model.map(lambda m: MODEL_INFO[m][0])
P["size"] = P.model.map(lambda m: MODEL_INFO[m][1])
P["correct"] = (P.pred == P.label).astype(int)
P["correct_raw"] = (P.pred_raw == P.label).astype(int)

rows = []
for (m, t), g in P.groupby(["model", "task"]):
    lp = g[[f"lp{k}" for k in range(K[t])]].to_numpy(float)
    truth = DATA[t][["item_id", "cond", "label"]].merge(g[["item_id", "cond", "label"]], on=["item_id", "cond"], how="outer",
                                                         suffixes=("", "_run"))
    rows.append({"model": SHORT[m], "task": t, "rows": len(g), "expected": len(DATA[t]),
                 "duplicates": int(g.duplicated(["item_id", "cond"]).sum()), "finite": bool(np.isfinite(lp).all()),
                 "pred_raw_reproduced": float((lp.argmax(1) == g.pred_raw.values).mean()),
                 "labels_match": bool((truth.label == truth.label_run).all()),
                 "nll_nonneg": bool((g.filter(like="nll_").dropna(axis=1, how="all") >= 0).all().all())})
integ = pd.DataFrame(rows)
integ.to_csv(f"{OUT}/checks/integrity.csv", index=False)
bad = integ[(integ.rows != integ.expected) | (integ.duplicates > 0) | ~integ.finite | (integ.pred_raw_reproduced < 1)
            | ~integ.labels_match | ~integ.nll_nonneg]
require("all prediction files complete and consistent", bad.empty, bad.to_dict("records")[:3])

In [ ]:
rng = np.random.default_rng(SEED)
N_ITEMS = {t: DATA[t].item_id.nunique() for t in TASKS}
IDX = {t: rng.integers(0, N_ITEMS[t], (B, N_ITEMS[t])) for t in TASKS}   # shared resamples: paired comparisons

def acc_matrix(g, col="correct"):
    return g.pivot(index="item_id", columns="cond", values=col).sort_index()[CONDITIONS].to_numpy(float)

rows = []
for (m, t), g in P.groupby(["model", "task"]):
    cm = acc_matrix(g)
    boot = cm[IDX[t], 0].mean(1)
    gg = GEN[(GEN.model == m) & (GEN.task == t)]
    clean = g[g.cond == "clean"]
    r = {"model": m, "task": t, "family": MODEL_INFO[m][0], "size": MODEL_INFO[m][1],
         "prefill": g.prefill.iloc[0], "letter_mass": clean.letter_mass.median(),
         "gen_agree": (gg.parsed == gg.pred_raw).mean(), "gen_parsed": (gg.parsed >= 0).mean(),
         "acc_clean": cm[:, 0].mean(), "acc_lo": np.percentile(boot, 2.5), "acc_hi": np.percentile(boot, 97.5)}
    reasons = []
    if r["letter_mass"] < MIN_LETTER_MASS: reasons.append("letter mass")
    if r["gen_agree"] < MIN_GEN_AGREEMENT: reasons.append("generation disagrees")
    if r["acc_clean"] - CHANCE[t] < MIN_ABOVE_CHANCE or r["acc_lo"] <= CHANCE[t]: reasons.append("near chance")
    r["valid"], r["excluded_because"] = not reasons, ", ".join(reasons)
    rows.append(r)
VALID = pd.DataFrame(rows)
VALID.to_csv(f"{OUT}/checks/validity.csv", index=False)
OK = set(zip(VALID[VALID.valid].model, VALID[VALID.valid].task))
P["valid"] = [(m, t) in OK for m, t in zip(P.model, P.task)]
KEY["valid_pairs"] = f"{len(OK)}/{len(VALID)}"
print(f"valid (model, task) pairs: {len(OK)} of {len(VALID)}")
ex = VALID[~VALID.valid].assign(model=lambda d: d.model.map(SHORT))[["model", "task", "letter_mass", "gen_agree", "acc_clean", "excluded_because"]]
print("excluded:"); display(ex.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 0.3 * len(MODELS) + 1.6), sharey=True)
for ax, k, title, vmin in [(axes[0], "letter_mass", "a. Letter mass (clean)", 0), (axes[1], "gen_agree", "b. Free answer = scored answer", 0),
                           (axes[2], "acc_clean", "c. Clean accuracy (× = excluded)", 0)]:
    M = VALID.pivot(index="model", columns="task", values=k).reindex(index=MODELS, columns=TASKS)
    ax.imshow(M.values, cmap=SEQ, vmin=vmin, vmax=1, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.6 else "#2b2a28")
            if k == "acc_clean" and (MODELS[i], TASKS[j]) not in OK:
                ax.text(j + 0.38, i - 0.25, "×", color="#e34948", fontsize=9, ha="center", va="center")
    ax.set(xticks=range(len(TASKS)), xticklabels=TASKS, yticks=range(len(MODELS)), yticklabels=[SHORT[m] for m in MODELS], title=title)
    ax.grid(False)
plt.tight_layout()
save(fig, "fig04_validity", "Is each model doing the task? Letter mass, agreement of free answers with scored answers, and clean accuracy; × marks pairs excluded by the validity gate.")

## Step 6 · RQ1 — How much does noise hurt?

- **drop** = clean accuracy − noisy accuracy (percentage points).
- **normalised drop** = drop ÷ (clean accuracy − chance): the share of the model's above-chance skill that the noise removes (0 = no loss, 1 = back to chance). It makes tasks with different chance levels comparable.
- 95% CIs: paired bootstrap over items (B = 2000, the same resamples for every model of a task).
- Means across models use **valid pairs only**.

In [ ]:
def ci(a):
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) else (np.nan, np.nan)

rows, BOOT = [], {}
for (m, t), g in P.groupby(["model", "task"]):
    cm = acc_matrix(g)
    bs = cm[IDX[t]].mean(1)   # (B, conditions)
    nll = g.assign(n=g.filter(like="nll_").sum(axis=1)).pivot(index="item_id", columns="cond", values="n").sort_index()[CONDITIONS]
    for j, c in enumerate(CONDITIONS):
        ca, a = cm[:, 0].mean(), cm[:, j].mean()
        above = ca - CHANCE[t]
        nd_b = (bs[:, 0] - bs[:, j]) / np.where(bs[:, 0] - CHANCE[t] > 0.05, bs[:, 0] - CHANCE[t], np.nan)
        BOOT[(m, t, c)] = nd_b
        lo, hi = ci(nd_b)
        dlo, dhi = ci(bs[:, 0] - bs[:, j])
        rows.append({"model": m, "task": t, "cond": c, "family": MODEL_INFO[m][0], "size": MODEL_INFO[m][1],
                     "valid": (m, t) in OK, "acc": a, "drop": ca - a, "drop_lo": dlo, "drop_hi": dhi,
                     "norm_drop": (ca - a) / above if above > 0.05 else np.nan, "nd_lo": lo, "nd_hi": hi,
                     "acc_raw": g[g.cond == c].correct_raw.mean(), "letter_mass": g[g.cond == c].letter_mass.median(),
                     "nll_sum": nll[c].sum()})
S = pd.DataFrame(rows)
ts_m = ts.groupby(["model", "task", "cond"])[["ntok", "nsyl", "nbroken"]].sum()
S = S.merge(pd.DataFrame({"fertility": ts_m.ntok / ts_m.nsyl, "broken_rate": ts_m.nbroken / ts_m.nsyl}).reset_index(),
            on=["model", "task", "cond"], how="left")
base = S[S.cond == "clean"].set_index(["model", "task"])
syl_clean = ts[ts.cond == "clean"].groupby(["model", "task"]).nsyl.sum()
keys = list(zip(S.model, S.task))
S["d_fertility"] = S.fertility.values - base.fertility.reindex(keys).values
S["d_broken"] = S.broken_rate.values - base.broken_rate.reindex(keys).values
S["d_nll_syl"] = (S.nll_sum.values - base.nll_sum.reindex(keys).values) / syl_clean.reindex(keys).values
S.to_csv(f"{OUT}/tables/summary_all.csv", index=False)
V = S[S.valid]

# Table: clean accuracy and normalised drop under strip, with CIs
def cell(r):
    return f"{r.norm_drop:.2f} [{r.nd_lo:.2f}, {r.nd_hi:.2f}]" if np.isfinite(r.norm_drop) else "--"
T = S[S.cond == "strip"].assign(cell=lambda d: [cell(r) if r.valid else "excl." for r in d.itertuples()])
tab_main = T.pivot(index="model", columns="task", values="cell").reindex(index=MODELS, columns=TASKS)
tab_main.index = [SHORT[m] for m in MODELS]
tab_clean = S[S.cond == "clean"].pivot(index="model", columns="task", values="acc").reindex(index=MODELS, columns=TASKS)
tab_clean.index = [SHORT[m] for m in MODELS]
save_table(tab_clean, "table_clean_accuracy", "Clean accuracy (calibrated).")
save_table(tab_main, "table_strip_norm_drop", "Normalised drop under full diacritic removal, 95% paired bootstrap CI.")
print("Clean accuracy (chance:", {t: round(c, 2) for t, c in CHANCE.items()}, ")")
display(tab_clean.round(2))
print("Normalised drop under strip [95% CI]:")
display(tab_main)

In [ ]:
# Mean normalised drop per task × condition over valid models, with a bootstrap CI over items (paired, models averaged)
rows = []
for t in TASKS:
    ms = [m for m in MODELS if (m, t) in OK]
    for c in NOISY:
        if not ms:
            continue
        b = np.nanmean(np.stack([BOOT[(m, t, c)] for m in ms]), axis=0)
        lo, hi = ci(b)
        rows.append({"task": t, "cond": c, "models": len(ms),
                     "mean_norm_drop": np.nanmean([S[(S.model == m) & (S.task == t) & (S.cond == c)].norm_drop.iloc[0] for m in ms]),
                     "lo": lo, "hi": hi})
AGG = pd.DataFrame(rows, columns=["task", "cond", "models", "mean_norm_drop", "lo", "hi"])
save_table(AGG.set_index(["task", "cond"]), "table_mean_drop", "Mean normalised drop over valid models, 95% CI.")
KEY["mean_norm_drop"] = AGG.pivot(index="task", columns="cond", values="mean_norm_drop").round(3).to_dict()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
for i, t in enumerate(TASKS):
    a = AGG[(AGG.task == t) & AGG.cond.isin(LEVEL)]
    x = [0] + [LEVEL[c] for c in a.cond]
    ax.plot(x, [0] + list(a.mean_norm_drop), marker="o", color=PALETTE[i], label=TASK_NAME[t])
    ax.fill_between(x, [0] + list(a.lo), [0] + list(a.hi), color=PALETTE[i], alpha=0.12, lw=0)
ax.set(xlabel="% of accented words stripped", ylabel="normalised drop (0 = none, 1 = chance)", xticks=[0, 25, 50, 75, 100],
       title="a. Dose-response: more stripping, more loss")
ax.legend(fontsize=7)
ax = axes[1]
x = np.arange(len(TASKS))
for j, c in enumerate(["strip", "teencode", "mixed"]):
    a = AGG[AGG.cond == c].set_index("task").reindex(TASKS)
    ax.bar(x + (j - 1) * 0.26, a.mean_norm_drop, 0.24, color=COND_COLOR[c], label=c,
           yerr=[a.mean_norm_drop - a.lo, a.hi - a.mean_norm_drop], error_kw=dict(lw=1, ecolor="#52514e"))
ax.axhline(0, color="#8a8984", lw=1)
ax.set(xticks=x, xticklabels=TASKS, ylabel="normalised drop", title="b. Diacritics vs teen code")
ax.legend(fontsize=7)
plt.tight_layout()
save(fig, "fig05_effect", "Mean normalised drop over valid models (95% paired bootstrap CI): a. by stripping level; b. full stripping vs teen code vs both.")

In [ ]:
# Every valid model: normalised drop under strip, per task (forest plot)
fig, axes = plt.subplots(1, len(TASKS), figsize=(3.1 * len(TASKS), 0.28 * len(MODELS) + 1.4), sharey=True)
for ax, t in zip(axes, TASKS):
    for i, m in enumerate(MODELS):
        r = S[(S.model == m) & (S.task == t) & (S.cond == "strip")].iloc[0]
        if not r.valid:
            ax.text(0.5, i, "excluded", fontsize=6, color="#8a8984", va="center", ha="center")
            continue
        ax.errorbar(r.norm_drop, i, xerr=[[r.norm_drop - r.nd_lo], [r.nd_hi - r.norm_drop]], fmt="o",
                    color=FAM_COLOR[r.family], ms=4, lw=1.2, capsize=0)
    ax.axvline(0, color="#8a8984", lw=1, ls=":")
    ax.axvline(1, color="#8a8984", lw=1, ls=":")
    ax.set(title=TASK_NAME[t], xlabel="normalised drop (strip)", xlim=(-0.3, 1.3), yticks=range(len(MODELS)),
           yticklabels=[SHORT[m] for m in MODELS])
axes[0].invert_yaxis()
plt.tight_layout()
save(fig, "fig06_forest_strip", "Normalised drop under full diacritic removal for every valid (model, task), 95% CI; colour = model family.")

In [ ]:
# Robustness of the effect: calibration choice, macro-F1, generation-based accuracy
from sklearn.metrics import f1_score
rows = []
for (m, t), g in P[P.valid].groupby(["model", "task"]):
    lp = g[[f"lp{k}" for k in range(K[t])]].to_numpy(float)
    p = softmax_rows(lp)
    percond = np.empty(len(g), int)
    for c in CONDITIONS:
        i = (g.cond == c).to_numpy()
        percond[i] = (np.log(p[i]) - np.log(p[i].mean(0))).argmax(1)
    g = g.assign(pred_percond=percond)
    def nd(col, metric="acc"):
        f = (lambda d: (d[col] == d.label).mean()) if metric == "acc" else (lambda d: f1_score(d.label, d[col], average="macro"))
        c0, c1 = f(g[g.cond == "clean"]), f(g[g.cond == "strip"])
        ch = CHANCE[t] if metric == "acc" else 1 / K[t]
        return (c0 - c1) / (c0 - ch) if c0 - ch > 0.05 else np.nan
    gg = GEN[(GEN.model == m) & (GEN.task == t)]
    ga = gg.assign(ok=gg.parsed == gg.label).groupby("cond").ok.mean()
    rows.append({"model": m, "task": t, "dev-prior calibration (main)": nd("pred"), "no calibration": nd("pred_raw"),
                 "per-condition calibration": nd("pred_percond"), "macro-F1 (main calibration)": nd("pred", "f1"),
                 "free generation (subset)": (ga["clean"] - ga["strip"]) / (ga["clean"] - CHANCE[t]) if ga["clean"] - CHANCE[t] > 0.05 else np.nan})
ROB = pd.DataFrame(rows)
rob = ROB.groupby("task").mean(numeric_only=True).reindex(TASKS)
save_table(rob, "table_robustness", "Mean normalised drop under strip with other scoring choices (valid pairs).")
KEY["robustness_strip"] = rob.round(3).to_dict()
print("Mean normalised drop under strip, by scoring choice (valid pairs):")
display(rob.round(2))

## Step 7 · RQ2 — Which models are more robust?

- **Size within a family** (same tokenizer, same training recipe): smallest vs largest model, difference in normalised drop under `strip`, paired bootstrap CI, **Holm-corrected** p-values over all (family × task) tests.
- **Vietnamese-adapted vs general** models (SeaLLMs, VinaLLaMA vs the rest), descriptive.

In [ ]:
def boot_p(diff):
    d = diff[np.isfinite(diff)]
    return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())) if len(d) else np.nan

def holm(ps):
    ps = np.asarray(ps, float)
    order = np.argsort(ps)
    adj = np.empty_like(ps)
    run = 0
    for rank, i in enumerate(order):
        run = max(run, (len(ps) - rank) * ps[i])
        adj[i] = min(1.0, run)
    return adj

rows = []
for fam in FAMILIES:
    for t in TASKS:
        ms = sorted([m for m in MODELS if MODEL_INFO[m][0] == fam and (m, t) in OK], key=lambda m: MODEL_INFO[m][1])
        if len(ms) < 2:
            continue
        small, large = ms[0], ms[-1]
        d = BOOT[(small, t, "strip")] - BOOT[(large, t, "strip")]
        lo, hi = ci(d)
        rows.append({"family": fam, "task": t, "small": SHORT[small], "large": SHORT[large],
                     "nd_small": S[(S.model == small) & (S.task == t) & (S.cond == "strip")].norm_drop.iloc[0],
                     "nd_large": S[(S.model == large) & (S.task == t) & (S.cond == "strip")].norm_drop.iloc[0],
                     "diff (small − large)": np.nanmean(d), "lo": lo, "hi": hi, "p": boot_p(d)})
SCALE = pd.DataFrame(rows, columns=["family", "task", "small", "large", "nd_small", "nd_large", "diff (small − large)", "lo", "hi", "p"])
if len(SCALE):
    SCALE["p_holm"] = holm(SCALE.p.values)
    SCALE["larger is more robust"] = (SCALE.lo > 0) & (SCALE.p_holm < 0.05)
    save_table(SCALE.set_index(["family", "task"]), "table_scale", "Smallest vs largest model per family (normalised drop under strip); Holm-corrected.")
    display(SCALE.round(3))
KEY["scale"] = SCALE.round(4).to_dict("records")

fig, axes = plt.subplots(1, len(TASKS), figsize=(3.1 * len(TASKS), 3.2), sharey=True)
for ax, t in zip(axes, TASKS):
    for fam in FAMILIES:
        g = S[(S.family == fam) & (S.task == t) & (S.cond == "strip") & S.valid].sort_values("size")
        if len(g) < 2:
            continue
        ax.errorbar(g["size"], g.norm_drop, yerr=[g.norm_drop - g.nd_lo, g.nd_hi - g.norm_drop], marker="o",
                    color=FAM_COLOR[fam], label=fam, capsize=2, lw=1.6, elinewidth=0.8)
    ax.set(title=TASK_NAME[t], xlabel="parameters (B)", xscale="log", xticks=[1.5, 3, 7], xticklabels=["1.5", "3", "7"])
    ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    ax.axhline(1, color="#8a8984", lw=1, ls=":")
    ax.set_ylim(-0.4, 1.6)   # CIs beyond this range are clipped
axes[0].set_ylabel("normalised drop (strip)")
h, l = axes[0].get_legend_handles_labels()
for ax in axes[1:]:
    hh, ll = ax.get_legend_handles_labels()
    h += [x for x, y in zip(hh, ll) if y not in l]
    l += [y for y in ll if y not in l]
if l:
    fig.legend(h, l, loc="lower center", ncol=len(l), fontsize=8, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
save(fig, "fig07_scale", "Normalised drop under strip against model size; one line per family (same tokenizer along a line), 95% CI.")

In [ ]:
s = V[V.cond == "strip"].assign(vi=lambda d: d.model.map(lambda m: MODEL_INFO[m][2]))
vi_tab = s.groupby(["task", "vi"]).norm_drop.agg(["mean", "count"]).unstack("vi")
vi_tab.columns = [f"{a} ({'Vietnamese-adapted' if b else 'general'})" for a, b in vi_tab.columns]
save_table(vi_tab.reindex(TASKS), "table_vi_adapted", "Normalised drop under strip: Vietnamese-adapted vs general models (descriptive).")
display(vi_tab.reindex(TASKS).round(2))

## Step 8 · RQ3 — Tokenizer or exposure?

Two predictors of the loss, both measured **on the exact tokens the model read**:
- **tokenizer:** Δ tokens per syllable (`d_tok`) and Δ share of broken syllables (`d_broken`) between noisy and clean;
- **exposure:** Δ NLL per syllable (`d_nll`): how much more surprising the noisy input is to the model.

**Model level:** within each condition (so "more noise" does not drive the correlation), across valid (model, task) points.

**Item level (main test):** one row per (model, task, item, noisy condition) where the model was right on the clean item and the noise changed the item; outcome = flip to wrong. A **linear probability model with item × condition fixed effects and model fixed effects** compares different models on the *same noisy item*. Within a group of models that share a tokenizer, `d_tok` and `d_broken` are identical for every model, so any remaining effect of `d_nll` cannot be a tokenizer effect. SEs are clustered by article (ViMMRC), premise (XNLI) or item.

In [ ]:
from scipy.stats import spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf

gs = V[V.cond != "clean"].dropna(subset=["norm_drop", "d_nll_syl"]).copy()
PRED = ["d_fertility", "d_broken", "d_nll_syl"]
rows = []
for c in NOISY:
    g = gs[gs.cond == c]
    if len(g) >= 5:
        rows.append({"cond": c, "points": len(g), **{k: spearmanr(g[k], g.norm_drop)[0] for k in PRED}})
RHO = pd.DataFrame(rows, columns=["cond", "points"] + PRED).set_index("cond")
save_table(RHO, "table_model_level_rho", "Spearman ρ between normalised drop and each predictor, within condition.")
print("Model level, Spearman ρ with normalised drop (within condition):")
display(RHO.round(2))
if len(gs) > 10:
    for k in PRED:
        gs[f"z_{k}"] = (gs[k] - gs[k].mean()) / gs[k].std()
    f = smf.ols("norm_drop ~ z_d_fertility + z_d_broken + z_d_nll_syl + C(cond) + C(task)", data=gs).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(gs.model)[0]})
    ML = pd.DataFrame({"coef": f.params, "lo": f.conf_int()[0], "hi": f.conf_int()[1], "p": f.pvalues}).loc[
        ["z_d_fertility", "z_d_broken", "z_d_nll_syl"]]
    save_table(ML, "table_model_level_ols", "Model-level OLS (condition and task fixed effects, SE clustered by model).", "%.3f")
    KEY["model_level_ols"] = ML.round(4).reset_index().to_dict("records")
    display(ML.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
g = gs[gs.cond == "strip"]
for ax, k, lab in [(axes[0], "d_fertility", "Δ tokens per syllable (strip − clean)"),
                   (axes[1], "d_nll_syl", "Δ NLL per syllable (strip − clean)")]:
    for fam, gf in g.groupby("family"):
        ax.scatter(gf[k], gf.norm_drop, s=30, color=FAM_COLOR[fam], label=fam, edgecolor="white", lw=0.6)
    r = spearmanr(g[k], g.norm_drop)[0] if len(g) > 3 else np.nan
    ax.set(xlabel=lab, title=f"Spearman ρ = {r:.2f} ({len(g)} model × task, strip)")
axes[0].set_ylabel("normalised drop (strip)")
axes[1].legend(fontsize=7)
plt.tight_layout()
save(fig, "fig08_model_level", "Model level, full stripping: loss against tokenizer change (left) and against surprise (right).")

In [ ]:
F = P.merge(ts[["model", "task", "item_id", "cond", "ntok", "nsyl", "nbroken"]], on=["model", "task", "item_id", "cond"])
F["nll"] = F.filter(like="nll_").sum(axis=1)
F = F.merge(FEAT, on=["task", "item_id", "cond"])
base = F[F.cond == "clean"][["model", "task", "item_id", "correct", "ntok", "nsyl", "nbroken", "nll"]]
X = F[(F.cond != "clean") & F.valid].merge(base, on=["model", "task", "item_id"], suffixes=("", "_c"))
X = X[(X.correct_c == 1) & (X.changed > 0) & (X.nsyl_c > 0)].copy()
X["flip"] = 1 - X.correct
X["d_tok"] = (X.ntok - X.ntok_c) / X.nsyl_c
X["d_broken"] = X.nbroken / X.nsyl.clip(lower=1) - X.nbroken_c / X.nsyl_c
X["d_nll"] = (X.nll - X.nll_c) / X.nsyl_c
X["tok_class"] = X.model.map(TOK_CLASS)
X["item_cond"] = X.task + "|" + X.item_id.astype(str) + "|" + X.cond
X.to_csv(f"{OUT}/tables/item_level.csv.gz", index=False)
print(f"{len(X)} item-level rows, flip rate {X.flip.mean():.3f}")
display(X[["d_tok", "d_broken", "d_nll"]].corr().round(2))

def demean(df, cols, groups, iters=200, tol=1e-9):
    out = df[cols].astype(float).copy()
    for _ in range(iters):
        delta = 0
        for gcol in groups:
            mns = out.groupby(df[gcol].values).transform("mean")
            out -= mns
            delta = max(delta, float(np.abs(mns.to_numpy()).max()))
        if delta < tol:
            break
    return out

def fe_lpm(df, xs, by_tok):
    df = df.copy()
    df["grp"] = (df.tok_class + "|" + df.item_cond) if by_tok else df.item_cond
    df = df[df.grp.map(df.grp.value_counts()) > 1]
    if len(df) < 50 or df.model.nunique() < 2:
        return None, df
    dm = demean(df, ["flip"] + xs, ["grp", "model"])
    sd = df[xs].std().replace(0, np.nan)
    Z = dm[xs] / sd
    keep = [x for x in xs if np.isfinite(sd[x]) and Z[x].abs().sum() > 1e-9]
    if not keep:
        return None, df
    return sm.OLS(dm.flip, Z[keep]).fit(cov_type="cluster", cov_kwds={"groups": df.unit}), df

rows, binned = [], {}
same = X[X.tok_class.map(X.groupby("tok_class").model.nunique()) > 1]
SAMPLES = [("all valid models", X, False, ["d_nll", "d_tok", "d_broken"]),
           ("same tokenizer only", same, True, ["d_nll"])]
SAMPLES += [(f"task: {t}", X[X.task == t], False, ["d_nll", "d_tok", "d_broken"]) for t in TASKS]
SAMPLES += [(f"condition: {c}", X[X.cond == c], False, ["d_nll", "d_tok", "d_broken"]) for c in ["partial50", "strip", "teencode"]]
for name, df, by_tok, xs in SAMPLES:
    r, used = fe_lpm(df, xs, by_tok)
    if r is None:
        continue
    for k in r.params.index:
        rows.append({"sample": name, "term": k, "pp_per_SD": 100 * r.params[k], "lo": 100 * r.conf_int().loc[k, 0],
                     "hi": 100 * r.conf_int().loc[k, 1], "p": r.pvalues[k], "rows": len(used)})
    if name in ("all valid models", "same tokenizer only"):
        dm = demean(used, ["flip", "d_nll"], ["grp", "model"])
        binned[name] = (dm.d_nll / used.d_nll.std(), dm.flip)
FE = pd.DataFrame(rows, columns=["sample", "term", "pp_per_SD", "lo", "hi", "p", "rows"])
save_table(FE.set_index(["sample", "term"]), "table_item_level_fe",
           "Item-level linear probability model with item×condition and model fixed effects: change in flip probability (pp) per SD.")
KEY["item_level_fe"] = FE.round(4).to_dict("records")
display(FE.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
sub = FE[FE["sample"].isin(["all valid models", "same tokenizer only"] + [f"task: {t}" for t in TASKS])]
names = {"d_nll": "Δ NLL (exposure)", "d_tok": "Δ tokens", "d_broken": "Δ broken syllables"}
labels = list(dict.fromkeys(sub["sample"]))
for j, (term, col) in enumerate([("d_nll", "#2a78d6"), ("d_tok", "#eb6834"), ("d_broken", "#1baf7a")]):
    s_ = sub[sub.term == term].set_index("sample").reindex(labels)
    y = np.arange(len(labels)) + (j - 1) * 0.22
    ax.errorbar(s_.pp_per_SD, y, xerr=[s_.pp_per_SD - s_.lo, s_.hi - s_.pp_per_SD], fmt="o", color=col, ms=4,
                lw=1.2, label=names[term])
ax.axvline(0, color="#8a8984", lw=1)
ax.set(yticks=range(len(labels)), yticklabels=labels, xlabel="Δ flip probability (pp) per SD, same noisy item",
       title="a. What predicts a flip on the same item?")
ax.invert_yaxis()
ax.legend(fontsize=7, loc="lower right")
ax = axes[1]
for (name, (xv, yv)), col in zip(binned.items(), ["#2a78d6", "#eb6834"]):
    q = pd.qcut(xv, 10, duplicates="drop")
    b = pd.DataFrame({"x": xv, "y": yv}).groupby(q, observed=True).mean()
    ax.plot(b.x, 100 * b.y, marker="o", color=col, label=name)
ax.axhline(0, color="#8a8984", lw=1, ls=":")
ax.set(xlabel="Δ NLL within item × condition (SD)", ylabel="flip rate within item × condition (pp)",
       title="b. The model that finds the item\nmore surprising flips more")
ax.legend(fontsize=7)
plt.tight_layout()
save(fig, "fig09_mechanism", "Item-level two-way fixed-effects model: a. coefficients (95% CI, clustered); b. binned within-item relation between surprise and flips.")

In [ ]:
# Lexical ambiguity: are flips more likely when the stripped words had many accented readings?
A = X[X.cond.isin(["partial25", "partial50", "partial75", "strip"])].dropna(subset=["amb_bits"]).copy()
if len(A) > 100:
    for k in ["amb_bits", "d_nll", "d_tok"]:
        A[f"z_{k}"] = (A[k] - A[k].mean()) / A[k].std()
    fa = smf.ols("flip ~ z_amb_bits + z_d_nll + z_d_tok + C(model) + C(task) + C(cond)", data=A).fit(
        cov_type="cluster", cov_kwds={"groups": A.unit})
    AMB = pd.DataFrame({"pp_per_SD": 100 * fa.params, "lo": 100 * fa.conf_int()[0], "hi": 100 * fa.conf_int()[1],
                        "p": fa.pvalues}).loc[["z_amb_bits", "z_d_nll", "z_d_tok"]]
    save_table(AMB, "table_ambiguity", "Flip probability and lexical ambiguity of stripped words (LPM, fixed effects).", "%.3f")
    KEY["ambiguity"] = AMB.round(4).reset_index().to_dict("records")
    display(AMB.round(3))

## Step 9 · What do models do when they fail?

When noise removes skill, do predictions spread out (random errors) or **collapse onto one label** (systematic bias)? Shift in the predicted label distribution, clean → strip, valid pairs pooled per task.

In [ ]:
rows = []
for t in TASKS:
    g = P[(P.task == t) & P.valid]
    if g.empty:
        continue
    for c in ["clean", "strip"]:
        for col, kind in [("pred_raw", "uncalibrated"), ("pred", "calibrated")]:
            share = g[g.cond == c][col].value_counts(normalize=True).reindex(range(K[t]), fill_value=0)
            rows += [{"task": t, "cond": c, "kind": kind, "label": k, "share": v} for k, v in share.items()]
SHIFT = pd.DataFrame(rows)
fig, axes = plt.subplots(1, len(TASKS), figsize=(3.1 * len(TASKS), 3), sharey=True)
for ax, t in zip(axes, TASKS):
    g = SHIFT[(SHIFT.task == t) & (SHIFT.kind == "uncalibrated")]
    if g.empty:
        ax.set_title(TASK_NAME[t] + "\n(no valid model)")
        continue
    w = g.pivot(index="label", columns="cond", values="share")
    x = np.arange(K[t])
    ax.bar(x - 0.2, w["clean"], 0.38, color="#86b6ef", label="clean")
    ax.bar(x + 0.2, w["strip"], 0.38, color="#0d366b", label="strip")
    ax.axhline(1 / K[t], color="#8a8984", lw=1, ls=":")
    labs = TASK_SPECS[t]["labels_en"] or list("ABCD")
    ax.set(xticks=x, xticklabels=[l.split(" (")[0] for l in labs], title=TASK_NAME[t])
    ax.tick_params(axis="x", rotation=45, labelsize=7)
axes[0].set_ylabel("share of predictions (uncalibrated)")
axes[0].legend(fontsize=7)
plt.tight_layout()
save(fig, "fig10_prediction_shift", "Predicted label distribution, clean vs strip, valid models pooled (uncalibrated); dotted = balanced truth.")

## Step 10 · Conclusions for the paper

Generated from the numbers above, so they always match the run. Read them together with the figures; the caveats at the end belong in the paper's *Limitations*.

In [ ]:
def fmt_ci(v, lo, hi):
    return f"{v:.2f} [95% CI {lo:.2f}, {hi:.2f}]"

lines = ["# Pilot 1A v4 — conclusions (auto-generated)", ""]
lines.append(f"**Setup.** {len(MODELS)} models, {len(TASKS)} tasks, {sum(N_ITEMS.values())} test items × {len(CONDITIONS)} conditions. "
             f"{KEY['valid_pairs']} (model, task) pairs passed the validity gate (letter mass ≥ {MIN_LETTER_MASS}, "
             f"free-answer agreement ≥ {MIN_GEN_AGREEMENT}, clean accuracy CI above chance).")
lines += ["", "## RQ1 — Effect"]
if len(AGG):
    st = AGG[AGG.cond == "strip"].set_index("task")
    for t in TASKS:
        if t in st.index:
            r = st.loc[t]
            lines.append(f"- **{TASK_NAME[t]}**: removing all diacritics removes {fmt_ci(r.mean_norm_drop, r.lo, r.hi)} of the models' "
                         f"above-chance skill ({int(r.models)} valid models).")
    tc = AGG[AGG.cond == "teencode"].set_index("task").mean_norm_drop
    lines.append(f"- **Teen code** alone costs little (mean normalised drop {tc.mean():.2f} across tasks): the damage comes from missing diacritics.")
    mono = []
    for (m, t), g in V[V.cond.isin(LEVEL)].groupby(["model", "task"]):
        a = g.set_index("cond").reindex(["clean", "partial25", "partial50", "partial75", "strip"]).acc.values
        mono.append(np.all(np.diff(a) <= 0.02))
    lines.append(f"- **Dose-response:** accuracy falls (near-)monotonically with the share of stripped words in {np.mean(mono):.0%} of valid pairs.")
    rb = rob.mean()
    lines.append(f"- **Robust to scoring choices:** mean normalised drop under strip is {rb['dev-prior calibration (main)']:.2f} with dev calibration, "
                 f"{rb['no calibration']:.2f} without calibration, {rb['macro-F1 (main calibration)']:.2f} in macro-F1, and "
                 f"{rb['free generation (subset)']:.2f} when the model's free answer is parsed instead of scored.")
lines += ["", "## RQ2 — Which models"]
if len(SCALE):
    sig = SCALE[SCALE["larger is more robust"]]
    lines.append(f"- **Size within a family:** the largest model is significantly more robust than the smallest in {len(sig)} of "
                 f"{len(SCALE)} (family, task) tests after Holm correction"
                 + (": " + ", ".join(f"{r.family}/{r.task} ({r.nd_small:.2f} → {r.nd_large:.2f})" for r in sig.itertuples()) if len(sig) else "") + ".")
    nsig_rev = SCALE[(SCALE.hi < 0) & (SCALE.p_holm < 0.05)]
    if len(nsig_rev):
        lines.append(f"- In {len(nsig_rev)} test(s) the larger model is significantly *less* robust: "
                     + ", ".join(f"{r.family}/{r.task}" for r in nsig_rev.itertuples()) + ".")
if len(vi_tab):
    lines.append("- **Vietnamese-adapted vs general models** (descriptive, see table_vi_adapted): "
                 + "; ".join(f"{t}: " + ", ".join(f"{c.split(' (')[1][:-1]} {v:.2f}" for c, v in vi_tab.loc[t].items() if c.startswith("mean") and np.isfinite(v))
                             for t in TASKS if t in vi_tab.index) + ".")
lines += ["", "## RQ3 — Why"]
if len(FE):
    def term(sample, k):
        r = FE[(FE["sample"] == sample) & (FE.term == k)]
        return None if r.empty else r.iloc[0]
    a, b_, c_ = term("all valid models", "d_nll"), term("all valid models", "d_tok"), term("all valid models", "d_broken")
    s_ = term("same tokenizer only", "d_nll")
    if a is not None:
        lines.append(f"- **Same noisy item, different models:** one SD more surprise (Δ NLL) raises the flip probability by "
                     f"{a.pp_per_SD:.1f} pp [{a.lo:.1f}, {a.hi:.1f}], p = {a.p:.3g}; Δ tokens {b_.pp_per_SD:.1f} pp (p = {b_.p:.2g}), "
                     f"Δ broken syllables {c_.pp_per_SD:.1f} pp (p = {c_.p:.2g}).")
    if s_ is not None:
        lines.append(f"- **Same tokenizer** (tokenization identical by construction): Δ NLL {s_.pp_per_SD:.1f} pp [{s_.lo:.1f}, {s_.hi:.1f}], p = {s_.p:.3g}. "
                     "An effect here cannot come from the tokenizer.")
if "model_level_ols" in KEY:
    ml = {r["index"]: r for r in KEY["model_level_ols"]}
    lines.append(f"- **Model level** (condition and task fixed effects): Δ NLL {ml['z_d_nll_syl']['coef']:.3f} (p = {ml['z_d_nll_syl']['p']:.2g}), "
                 f"Δ fertility {ml['z_d_fertility']['coef']:.3f} (p = {ml['z_d_fertility']['p']:.2g}), Δ broken {ml['z_d_broken']['coef']:.3f} (p = {ml['z_d_broken']['p']:.2g}).")
if "ambiguity" in KEY:
    am = KEY["ambiguity"][0]
    lines.append(f"- **Lexical ambiguity:** flips are {'more' if am['pp_per_SD'] > 0 else 'less'} likely when the stripped words have more accented readings "
                 f"({am['pp_per_SD']:.1f} pp per SD, p = {am['p']:.2g}).")
lines += ["", "## Limitations (write these in the paper)",
          "- Zero-shot multiple-choice prompting with one fixed option order; other prompt formats may change absolute accuracy.",
          "- Synthetic noise: real unaccented writing is not uniformly random (people drop diacritics on frequent words first).",
          "- Balanced samples of a few hundred items per task: CIs are wide for single models; pooled analyses are the reliable ones.",
          "- The exposure measure (NLL) is correlational: it shows *which model* suffers on *which item*, not why that model was trained that way.",
          "- Models 1.5–8B only; excluded (model, task) pairs are listed in checks/validity.csv."]
TEXT = "\n".join(lines)
open(f"{OUT}/conclusions.md", "w").write(TEXT)
display(Markdown(TEXT))

In [ ]:
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else round(float(o), 4)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o
json.dump(jsonable(KEY), open(f"{OUT}/key_numbers.json", "w"), ensure_ascii=False, indent=1)
pd.DataFrame(FIGS, columns=["figure", "caption"]).to_csv(f"{OUT}/figures/captions.csv", index=False)
print("checks:", sum(c["ok"] for c in CHECKS), "passed,", sum(not c["ok"] for c in CHECKS), "failed")
print("figures:")
for n, c in FIGS:
    print(f"  {n}: {c}")
print("\nEverything is in", OUT, "— when done: Runtime → Disconnect and delete runtime.")